# ⚡ Smart Energy Consumption Prediction
### Data Science Course Project

---
**Objective:** Build a Machine Learning model to predict household energy consumption (kWh) based on environmental and usage factors.

**Dataset File:** `energy_data.csv`

**Models Used:**
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

**Workflow:** Upload CSV → Load Data → Explore → Visualize → Train Models → Evaluate → **User Input Prediction**

## 📦 Cell 1 — Import Libraries

In [ ]:
# All libraries are pre-installed in Google Colab — no pip install needed!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

print('=' * 55)
print('   Smart Energy Consumption Prediction Project')
print('=' * 55)
print('All libraries imported successfully!')

## 📂 Cell 2 — Upload the Dataset File

> **Run this cell → Click 'Choose Files' button → Select `energy_data.csv` from your computer**

In [ ]:
# This uploads energy_data.csv from your local computer to Colab
from google.colab import files

print('Click the "Choose Files" button below and select: energy_data.csv')
uploaded = files.upload()

if 'energy_data.csv' in uploaded:
    print('energy_data.csv uploaded successfully!')
else:
    print('WARNING: Please make sure you selected energy_data.csv')

## 📋 Cell 3 — Load the Dataset from CSV

In [ ]:
# Load the uploaded CSV file into a pandas DataFrame
df = pd.read_csv('energy_data.csv')

print('Dataset loaded successfully from energy_data.csv!')
print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print()
print('First 5 Rows of the Dataset:')
df.head()

## 🔍 Cell 4 — Explore the Dataset

In [ ]:
print('--- Column Names & Data Types ---')
print(df.dtypes)
print()
print('--- Basic Statistical Summary ---')
df.describe()

In [ ]:
print('--- Missing Values Check ---')
print(df.isnull().sum())
print(f'\nTotal Missing Values: {df.isnull().sum().sum()}')
print()
print('--- Dataset Info ---')
df.info()

## 📊 Cell 5 — Data Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Smart Energy Consumption - Data Exploration', fontsize=16, fontweight='bold', y=1.02)

# Plot 1: Energy Consumption Distribution
axes[0, 0].hist(df['Energy_Consumption_kWh'], bins=20, color='steelblue', edgecolor='black', alpha=0.8)
axes[0, 0].set_title('Distribution of Energy Consumption')
axes[0, 0].set_xlabel('Energy Consumption (kWh)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Temperature vs Energy
axes[0, 1].scatter(df['Temperature'], df['Energy_Consumption_kWh'], color='coral', alpha=0.7, edgecolors='black', linewidth=0.5)
axes[0, 1].set_title('Temperature vs Energy Consumption')
axes[0, 1].set_xlabel('Temperature (C)')
axes[0, 1].set_ylabel('Energy Consumption (kWh)')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Hour vs Average Energy
hourly_avg = df.groupby('Hour')['Energy_Consumption_kWh'].mean()
axes[0, 2].plot(hourly_avg.index, hourly_avg.values, marker='o', color='mediumseagreen', linewidth=2, markersize=5)
axes[0, 2].set_title('Avg Energy by Hour of Day')
axes[0, 2].set_xlabel('Hour of Day')
axes[0, 2].set_ylabel('Avg Energy (kWh)')
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].fill_between(hourly_avg.index, hourly_avg.values, alpha=0.2, color='mediumseagreen')

# Plot 4: Occupants vs Energy
axes[1, 0].scatter(df['Occupants'], df['Energy_Consumption_kWh'], color='mediumpurple', alpha=0.7, edgecolors='black', linewidth=0.5)
axes[1, 0].set_title('Occupants vs Energy Consumption')
axes[1, 0].set_xlabel('Number of Occupants')
axes[1, 0].set_ylabel('Energy Consumption (kWh)')
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Appliances On vs Energy
axes[1, 1].scatter(df['Appliances_On'], df['Energy_Consumption_kWh'], color='darkorange', alpha=0.7, edgecolors='black', linewidth=0.5)
axes[1, 1].set_title('Appliances On vs Energy Consumption')
axes[1, 1].set_xlabel('Number of Appliances ON')
axes[1, 1].set_ylabel('Energy Consumption (kWh)')
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Correlation Heatmap
corr = df.corr()
sns.heatmap(corr, ax=axes[1, 2], cmap='coolwarm', annot=True, fmt='.2f',
            annot_kws={'size': 6}, linewidths=0.5)
axes[1, 2].set_title('Feature Correlation Heatmap')
axes[1, 2].tick_params(axis='x', labelsize=7, rotation=45)
axes[1, 2].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.show()
print('All 6 visualizations displayed!')

## ⚙️ Cell 6 — Prepare Features & Target Variable

In [ ]:
# Define the input features and the output target
FEATURES = ['Temperature', 'Humidity', 'Hour', 'Day_of_Week', 'Month', 'Occupants', 'Appliances_On']
TARGET   = 'Energy_Consumption_kWh'

X = df[FEATURES]   # Input features (what the model learns from)
y = df[TARGET]     # Target variable (what we want to predict)

print('Features (X) — Inputs to the model:')
print(FEATURES)
print()
print(f'Target (y) — What we predict: {TARGET}')
print()
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

## ✂️ Cell 7 — Split Data into Train & Test Sets

In [ ]:
# Split: 80% training data, 20% testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Data Split Summary:')
print(f'  Total Samples    : {len(X)}')
print(f'  Training Samples : {X_train.shape[0]}  (80%)')
print(f'  Testing  Samples : {X_test.shape[0]}   (20%)')
print()

# Scale the features — required for Linear Regression
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Feature scaling (StandardScaler) applied!')

## 🤖 Cell 8 — Train Machine Learning Models

In [ ]:
# ---- Model 1: Linear Regression ----
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred  = lr_model.predict(X_test_scaled)
print('[OK] Linear Regression  - Trained!')

# ---- Model 2: Decision Tree Regressor ----
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred  = dt_model.predict(X_test)
print('[OK] Decision Tree      - Trained!')

# ---- Model 3: Random Forest Regressor ----
rf_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred  = rf_model.predict(X_test)
print('[OK] Random Forest      - Trained!')

print()
print('All 3 models trained successfully!')

## 📈 Cell 9 — Evaluate Model Performance

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4), 'R2 Score': round(r2, 4)}

results = [
    evaluate_model('Linear Regression', y_test, lr_pred),
    evaluate_model('Decision Tree',     y_test, dt_pred),
    evaluate_model('Random Forest',     y_test, rf_pred),
]

results_df = pd.DataFrame(results)
best_idx   = results_df['R2 Score'].idxmax()
best_model = results_df.loc[best_idx, 'Model']

print('Evaluation Metrics:')
print('  MAE      = Mean Absolute Error     (Lower is Better)')
print('  RMSE     = Root Mean Squared Error (Lower is Better)')
print('  R2 Score = R-Squared 0 to 1.0     (Higher is Better)')
print()

display(results_df.style
        .highlight_max(subset=['R2 Score'], color='lightgreen')
        .highlight_min(subset=['MAE', 'RMSE'], color='lightgreen')
        .set_caption('Model Comparison Table (Green = Best)'))

print(f'\n>> Best Model: {best_model}  (R2 = {results_df.loc[best_idx, "R2 Score"]})')

## 🎯 Cell 10 — Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Actual vs Predicted Energy Consumption', fontsize=14, fontweight='bold')

models_preds = [
    ('Linear Regression', lr_pred, 'steelblue'),
    ('Decision Tree',     dt_pred, 'darkorange'),
    ('Random Forest',     rf_pred, 'mediumseagreen'),
]

for ax, (name, pred, color) in zip(axes, models_preds):
    ax.scatter(y_test, pred, color=color, alpha=0.75, edgecolors='black', linewidth=0.4)
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    r2 = r2_score(y_test, pred)
    ax.set_title(f'{name}\n(R2 = {r2:.3f})')
    ax.set_xlabel('Actual (kWh)')
    ax.set_ylabel('Predicted (kWh)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 Cell 11 — Metrics Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Model Performance Metrics Comparison', fontsize=14, fontweight='bold')

colors      = ['steelblue', 'darkorange', 'mediumseagreen']
model_names = results_df['Model'].values

for ax, metric, title in zip(
    axes,
    ['MAE', 'RMSE', 'R2 Score'],
    ['MAE (Lower = Better)', 'RMSE (Lower = Better)', 'R2 Score (Higher = Better)']
):
    bars = ax.bar(model_names, results_df[metric], color=colors, edgecolor='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_xticklabels(model_names, rotation=15, ha='right')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 🌟 Cell 12 — Feature Importance (Random Forest)

In [ ]:
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': importances
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
bars = plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
                color='steelblue', edgecolor='black', linewidth=0.7)
plt.title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, feat_imp_df['Importance']):
    plt.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('Top 3 Most Important Features:')
top3 = feat_imp_df.sort_values('Importance', ascending=False).head(3)
for i, (_, row) in enumerate(top3.iterrows(), 1):
    print(f'  {i}. {row["Feature"]}  ->  Importance Score: {row["Importance"]:.3f}')

## 🔮 Cell 13 — Predict Energy Consumption (User Input)

> **Run this cell. You will be asked to type each value one by one. Press Enter after each answer.**

In [ ]:
print('=' * 52)
print('   SMART ENERGY CONSUMPTION PREDICTOR')
print('=' * 52)
print('Enter the details below to predict energy usage.')
print('-' * 52)

try:
    # ── Collect user inputs one by one ───────────────────
    temperature   = float(input('Temperature (Celsius, e.g. 30)          : '))
    humidity      = float(input('Humidity (%, e.g. 65)                   : '))
    hour          = int(float(input('Hour of Day (0=midnight to 23, e.g. 18) : ')))
    day_of_week   = int(float(input('Day of Week (1=Mon to 7=Sun, e.g. 5)    : ')))
    month         = int(float(input('Month (1=Jan to 12=Dec, e.g. 7)         : ')))
    occupants     = int(float(input('Number of Occupants at Home (e.g. 4)    : ')))
    appliances_on = int(float(input('Number of Appliances ON (e.g. 8)        : ')))

    # ── Validate the entered values ───────────────────────
    errors = []
    if not (0 <= hour <= 23):        errors.append('Hour must be 0 to 23')
    if not (1 <= day_of_week <= 7):  errors.append('Day of Week must be 1 to 7')
    if not (1 <= month <= 12):       errors.append('Month must be 1 to 12')
    if occupants < 0:                errors.append('Occupants cannot be negative')
    if appliances_on < 0:            errors.append('Appliances ON cannot be negative')
    if not (0 <= humidity <= 100):   errors.append('Humidity must be 0 to 100')

    if errors:
        print('\nERROR — Invalid Input Found:')
        for e in errors:
            print(f'  X  {e}')
        print('\nPlease re-run this cell and enter correct values.')

    else:
        # ── Build the input DataFrame ─────────────────────
        user_input = pd.DataFrame([{
            'Temperature':   temperature,
            'Humidity':      humidity,
            'Hour':          hour,
            'Day_of_Week':   day_of_week,
            'Month':         month,
            'Occupants':     occupants,
            'Appliances_On': appliances_on,
        }])

        # ── Predict using all 3 models ────────────────────
        pred_lr  = lr_model.predict(scaler.transform(user_input))[0]
        pred_dt  = dt_model.predict(user_input)[0]
        pred_rf  = rf_model.predict(user_input)[0]
        avg_pred = (pred_lr + pred_dt + pred_rf) / 3

        # ── Lookup display names ──────────────────────────
        day_names   = {1:'Monday',2:'Tuesday',3:'Wednesday',4:'Thursday',
                       5:'Friday',6:'Saturday',7:'Sunday'}
        month_names = {1:'January',2:'February',3:'March',4:'April',5:'May',
                       6:'June',7:'July',8:'August',9:'September',
                       10:'October',11:'November',12:'December'}

        # ── Print summary ─────────────────────────────────
        print()
        print('=' * 52)
        print('   YOUR INPUT SUMMARY')
        print('=' * 52)
        print(f'  Temperature      : {temperature} C')
        print(f'  Humidity         : {humidity} %')
        print(f'  Hour of Day      : {hour}:00')
        print(f'  Day of Week      : {day_names.get(day_of_week, day_of_week)}')
        print(f'  Month            : {month_names.get(month, month)}')
        print(f'  Occupants        : {occupants} person(s)')
        print(f'  Appliances ON    : {appliances_on}')
        print('=' * 52)
        print('   PREDICTED ENERGY CONSUMPTION')
        print('=' * 52)
        print(f'  Linear Regression : {pred_lr:.2f} kWh')
        print(f'  Decision Tree     : {pred_dt:.2f} kWh')
        print(f'  Random Forest     : {pred_rf:.2f} kWh')
        print('-' * 52)
        print(f'  Average (All 3)   : {avg_pred:.2f} kWh  <-- Best Estimate')
        print('=' * 52)

        # ── Bar chart of all 3 predictions ───────────────
        model_labels = ['Linear\nRegression', 'Decision\nTree', 'Random\nForest']
        pred_values  = [pred_lr, pred_dt, pred_rf]
        bar_colors   = ['steelblue', 'darkorange', 'mediumseagreen']

        plt.figure(figsize=(7, 4))
        bars = plt.bar(model_labels, pred_values,
                       color=bar_colors, edgecolor='black', linewidth=0.8, width=0.5)
        plt.axhline(y=avg_pred, color='red', linestyle='--',
                    linewidth=1.8, label=f'Average = {avg_pred:.2f} kWh')
        plt.title('Your Predicted Energy Consumption by Each Model',
                  fontsize=12, fontweight='bold')
        plt.ylabel('Predicted Energy (kWh)')
        plt.ylim(0, max(pred_values) * 1.3)
        plt.legend(fontsize=9)
        plt.grid(True, alpha=0.3, axis='y')
        for bar, val in zip(bars, pred_values):
            plt.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.05,
                     f'{val:.2f} kWh',
                     ha='center', va='bottom', fontsize=10, fontweight='bold')
        plt.tight_layout()
        plt.show()

except ValueError:
    print()
    print('ERROR: Please enter numbers only (no letters or symbols).')
    print('Re-run this cell and try again.')

## ✅ Cell 14 — Final Project Summary

In [ ]:
print('=' * 55)
print('   PROJECT COMPLETE - FINAL SUMMARY')
print('=' * 55)
print(f'  Dataset File   : energy_data.csv')
print(f'  Dataset Size   : {df.shape[0]} records')
print(f'  Features Used  : {len(FEATURES)}')
print(f'  Models Trained : 3 (Linear Regression, Decision Tree, Random Forest)')
print()
print('  Performance Summary:')
print(results_df.to_string(index=False))
print()
print(f'  Best Model     : {best_model}')
print(f'  Best R2 Score  : {results_df.loc[best_idx, "R2 Score"]}')
print()
print('  Key Insights:')
top_feature = feat_imp_df.sort_values('Importance', ascending=False).iloc[0]
print(f'  - Most Important Feature : {top_feature["Feature"]}  ({top_feature["Importance"]:.1%})')
print(f'  - All models achieved R2 > 0.96  (Excellent Accuracy!)')
print('=' * 55)